# MemoryFlow — AI Embeddings & Face Grouping Pipeline

This Google Colab notebook processes your event collection using GPU acceleration (T4 GPU is free):
1. **CLIP ViT-B/32 Embeddings**: Computes 512-dim visual embeddings and calibrated text prototypes for scene/activity classification.
2. **InsightFace Face Grouping**: Detects faces, computes face embeddings, and clusters them with blank editable names.
3. **Export JSONs**: Generates `embeddings.json` and `face_groups.json` ready to upload to MemoryFlow.

### Step 0: Ensure GPU is Enabled
Go to **Runtime > Change runtime type > T4 GPU > Save**.

In [ ]:
# Install dependencies (approx 1 minute)
!pip install -q transformers torch torchvision pillow insightface onnxruntime-gpu scikit-learn requests tqdm
print('✓ Dependencies installed successfully!')

In [ ]:
import os
import json
import requests
from io import BytesIO
from PIL import Image
import numpy as np
import torch
from transformers import CLIPProcessor, CLIPModel
from sklearn.cluster import DBSCAN
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

### Step 1: Provide Your Media Items
You can either:
1. Export the list of media IDs from your running MemoryFlow `/api/collections/elevate/data`, OR
2. Upload a folder of photos directly to Colab.

Below, paste the MemoryFlow server collection URL or media list:

In [ ]:
# Enter your local or deployed MemoryFlow API URL, or leave as default to fetch from public collection
COLLECTION_API_URL = 'http://127.0.0.1:5173/api/collections/elevate/data'

# If running externally, you can also paste media list directly as a JSON string
MEDIA_ITEMS = []
try:
    res = requests.get(COLLECTION_API_URL, timeout=10)
    if res.status_code == 200:
        data = res.json()
        MEDIA_ITEMS = data.get('mediaItems', [])
        print(f'Loaded {len(MEDIA_ITEMS)} media items from MemoryFlow API.')
except Exception as e:
    print('Could not reach local server directly. You can upload an exported collection.json to Colab or enter Drive file IDs below.')

### Step 2: Compute CLIP Visual Embeddings & Text Prototypes

In [ ]:
# Load CLIP ViT-B/32
print('Loading CLIP model...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

# Activity labels to compute text prototypes for
ACTIVITY_LABELS = [
    'group photo', 'person speaking', 'stage performance', 'rehearsal',
    'preparation', 'auditorium', 'classroom', 'outdoors'
]

# Compute real CLIP text embeddings for scene classification
text_inputs = clip_processor(text=[f'a photo of a {label}' for label in ACTIVITY_LABELS], return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    activity_prototypes = {
        label: text_features[i].cpu().numpy().tolist()
        for i, label in enumerate(ACTIVITY_LABELS)
    }

print('✓ Calibrated activity text prototypes computed.')

In [ ]:
# Function to download thumbnail and extract 512-dim CLIP embedding
def get_clip_embedding(image_url):
    try:
        resp = requests.get(image_url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        if resp.status_code != 200:
            return None
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        inputs = clip_processor(images=img, return_tensors='pt').to(device)
        with torch.no_grad():
            image_features = clip_model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            return image_features[0].cpu().numpy().tolist()
    except Exception as e:
        return None

embeddings_output = {
    'model': 'openai/clip-vit-base-patch32',
    'dimension': 512,
    'prototypes': activity_prototypes,
    'items': []
}

for item in tqdm(MEDIA_ITEMS, desc='Processing CLIP embeddings'):
    url = item.get('thumbnailUrl') or item.get('lowResUrl')
    if not url:
        continue
    vec = get_clip_embedding(url)
    if vec:
        embeddings_output['items'].append({
            'mediaId': item['id'],
            'driveFileId': item.get('driveFileId'),
            'embedding': vec
        })

print(f'Computed {len(embeddings_output["items"])} visual embeddings.')
with open('embeddings.json', 'w') as f:
    json.dump(embeddings_output, f, indent=2)
print('✓ Saved embeddings.json')

### Step 3: Face Detection & Clustering with InsightFace

In [ ]:
import insightface
from insightface.app import FaceAnalysis

print('Initializing InsightFace...')
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

detected_faces = [] # list of { mediaId, embedding }

for item in tqdm(MEDIA_ITEMS, desc='Detecting faces'):
    url = item.get('thumbnailUrl') or item.get('lowResUrl')
    if not url:
        continue
    try:
        resp = requests.get(url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        if resp.status_code != 200:
            continue
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        img_np = np.array(img)[:, :, ::-1] # RGB to BGR
        faces = app.get(img_np)
        for face in faces:
            # Normalize face embedding
            feat = face.embedding / np.linalg.norm(face.embedding)
            detected_faces.append({
                'mediaId': item['id'],
                'driveFileId': item.get('driveFileId'),
                'embedding': feat
            })
    except Exception as e:
        continue

print(f'Detected {len(detected_faces)} total faces across collection.')

In [ ]:
# Cluster face embeddings with DBSCAN
face_groups_output = {
    'model': 'insightface/buffalo_l',
    'groups': []
}

if len(detected_faces) > 0:
    X = np.array([f['embedding'] for f in detected_faces])
    # Cosine distance metric for face clustering (eps=0.45 corresponds to approx 0.75 cosine similarity)
    clustering = DBSCAN(eps=0.45, min_samples=2, metric='cosine').fit(X)
    labels = clustering.labels_

    clusters = {}
    for idx, label in enumerate(labels):
        if label == -1:
            continue # Noise / unclustered single face
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(detected_faces[idx]['mediaId'])

    for group_idx, (cluster_id, media_ids) in enumerate(clusters.items()):
        unique_media = sorted(list(set(media_ids)))
        face_groups_output['groups'].append({
            'groupId': f'person_{group_idx + 1}',
            'label': '', # Blank editable label as requested
            'sampleMediaId': unique_media[0],
            'mediaIds': unique_media,
            'faceCount': len(media_ids)
        })

    print(f'✓ Formed {len(face_groups_output["groups"])} distinct person face groups.')
else:
    print('No faces found to cluster.')

with open('face_groups.json', 'w') as f:
    json.dump(face_groups_output, f, indent=2)
print('✓ Saved face_groups.json')

### Step 4: Download JSON Files to Import into MemoryFlow

In [ ]:
from google.colab import files
print('Downloading embeddings.json and face_groups.json...')
files.download('embeddings.json')
files.download('face_groups.json')
print('Done! Now upload these files into MemoryFlow > Admin > Import AI Data.')